# CS116 S6E10 - Model B1: Personal Travel Specialized Dual-GBDT (`B_personal_spec_v1`)

## 1. Động lực & Phân tích Tử huyệt:
- **Vấn đề cốt lõi:** Phân tích dữ liệu thực tế cho thấy:
  - Nhóm **Business Travel** (71% dữ liệu, ~497k dòng): Tỷ lệ hài lòng là **58.4%**, CV AUC mô hình đạt tới **0.9576**.
  - Nhóm **Personal Travel** (29% dữ liệu, ~202k dòng): Tỷ lệ hài lòng chỉ **9.7%**, và **CV AUC bị sụt giảm nghiêm trọng xuống chỉ 0.8336**!
- **Nguyên nhân:** Khi huấn luyện một mô hình GBDT toàn cục, 71% khách Business chi phối toàn bộ gradient và các điểm phân nhánh (tree splits). Hành vi của khách du lịch cá nhân (rất nhạy cảm với delay, chi phí, hành lý, checkin) bị mô hình coi như nhiễu hoặc dự đoán thiếu chính xác.
- **Giải pháp Dual-Specialist:**
  1. `Specialist_Personal`: Huấn luyện LightGBM/CatBoost riêng cho phân khúc Personal Travel với các đặc trưng tương tác độ trễ, tiện ích hành lý, và điều chỉnh trọng số nhãn (`scale_pos_weight` tối ưu cho 9.7% nhãn dương).
  2. `Specialist_Business`: Huấn luyện LightGBM riêng cho phân khúc Business Travel tập trung vào tiện ích cao cấp (Wifi, Inflight Entertainment, Leg room, Seat comfort).
  3. Ghép dự đoán sạch qua 5 fold của `data/folds.csv` để xuất `oof_B_personal_spec_v1.npy` và `test_B_personal_spec_v1.npy`.

In [1]:
import os
import sys
import time
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

# Đảm bảo import được module trong src/
sys.path.insert(0, os.path.abspath("../src"))
from config import DATA_DIR, OOF_DIR, SUB_DIR, TARGET, ID, CAT_COLS, NUM_COLS, SCORE_COLS
from data import load_data
from utils import save_preds

warnings.filterwarnings("ignore")
MODEL_NAME = "B_personal_spec_v1"
print(f"Huấn luyện mô hình: {MODEL_NAME}")

Huấn luyện mô hình: B_personal_spec_v1


## 2. Nạp dữ liệu chuẩn chung của nhóm (folds.csv)

In [2]:
train, test = load_data()
y_true = train[TARGET].values
folds = train["fold"].values

print(f"Train shape: {train.shape}, Test shape: {test.shape}")
print(f"Phân bố target toàn cục: {y_true.mean():.4f}")
print("Phân bố theo Type of Travel:")
print(train.groupby("Type of Travel")[TARGET].agg(["count", "mean"]))

Train shape: (699635, 24), Test shape: (299844, 22)
Phân bố target toàn cục: 0.4436
Phân bố theo Type of Travel:
                  count      mean
Type of Travel                   
Business travel  497441  0.584337
Personal Travel  202194  0.097263


## 3. Feature Engineering Chuyên Biệt cho Personal vs Business

In [3]:
def engineer_features(df):
    df = df.copy()
    
    # 1. Xử lý giá trị thiếu của Arrival Delay
    arr_delay = df["Arrival Delay in Minutes"].fillna(df["Departure Delay in Minutes"]).fillna(0)
    dep_delay = df["Departure Delay in Minutes"].fillna(0)
    
    # 2. Đặc trưng độ trễ tương tác
    df["Total_Delay"] = dep_delay + arr_delay
    df["Delay_Diff"] = arr_delay - dep_delay
    df["Has_Delay"] = (df["Total_Delay"] > 0).astype(int)
    df["Severe_Delay"] = (df["Total_Delay"] >= 30).astype(int)
    df["Delay_Per_Distance"] = df["Total_Delay"] / (df["Flight Distance"] + 1.0)
    
    # 3. Gom nhóm điểm số dịch vụ theo bản chất
    # Tiện ích cơ bản / mặt đất (Đặc biệt quan trọng với khách Personal Travel)
    df["Ground_Hygiene_Score"] = (
        df["Checkin service"] + df["Baggage handling"] + df["Cleanliness"] + df["Gate location"]
    ) / 4.0
    
    # Tiện ích bay cao cấp (Đặc biệt quan trọng với khách Business Travel)
    df["Flight_Comfort_Score"] = (
        df["Inflight wifi service"] + df["Inflight entertainment"] + 
        df["Seat comfort"] + df["Leg room service"] + df["On-board service"]
    ) / 5.0
    
    # Tiện ích số hóa / đặt vé
    df["Digital_Ease_Score"] = (
        df["Online boarding"] + df["Ease of Online booking"]
    ) / 2.0
    
    # Tổng hòa điểm hài lòng
    df["Mean_Survey_Score"] = df[SCORE_COLS].mean(axis=1)
    df["Min_Survey_Score"] = df[SCORE_COLS].min(axis=1)
    df["Zero_Score_Count"] = (df[SCORE_COLS] == 0).sum(axis=1)
    df["Top_Score_Count"] = (df[SCORE_COLS] == 5).sum(axis=1)
    
    # Tương tác độ tuổi & đường bay
    df["Age_Group"] = pd.cut(df["Age"], bins=[0, 25, 40, 60, 100], labels=[0, 1, 2, 3]).astype(float)
    df["Short_Haul"] = (df["Flight Distance"] < 800).astype(int)
    df["Long_Haul"] = (df["Flight Distance"] >= 2000).astype(int)
    
    return df

print("Tạo các feature chuyên biệt...")
train_fe = engineer_features(train)
test_fe = engineer_features(test)

# Chuyển các cột phân loại sang kiểu category để LightGBM xử lý trực tiếp
for c in CAT_COLS:
    train_fe[c] = train_fe[c].astype("category")
    test_fe[c] = test_fe[c].astype("category")

feature_cols = [c for c in train_fe.columns if c not in [ID, TARGET, "fold"]]
print(f"Tổng số feature đưa vào mô hình: {len(feature_cols)}")

Tạo các feature chuyên biệt...
Tổng số feature đưa vào mô hình: 36


## 4. Huấn luyện Dual-Specialist GBDT qua 5 Fold chuẩn

- Với mỗi fold $k$:
  1. Tách tập train thành 2 phân vùng: **Personal Travel** và **Business Travel**.
  2. `Model_Personal`: Tối ưu tham số riêng cho Personal Travel (`scale_pos_weight = 1.35`, `num_leaves = 31`, `learning_rate = 0.03`, `lambda_l2 = 20.0`).
  3. `Model_Business`: Tối ưu tham số cho Business Travel (`num_leaves = 63`, `learning_rate = 0.03`, `lambda_l2 = 10.0`).
  4. Dự đoán validation fold $k$ và tập test theo đúng nhánh tương ứng.

In [4]:
N_TRAIN = len(train)
N_TEST = len(test)

oof_preds = np.zeros(N_TRAIN, dtype=np.float32)
test_preds = np.zeros(N_TEST, dtype=np.float32)

is_personal_tr = (train_fe["Type of Travel"] == "Personal Travel").values
is_personal_te = (test_fe["Type of Travel"] == "Personal Travel").values

params_personal = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "learning_rate": 0.03,
    "num_leaves": 31,
    "max_depth": 6,
    "scale_pos_weight": 1.35,
    "subsample": 0.8,
    "colsample_bytree": 0.75,
    "reg_alpha": 1.0,
    "reg_lambda": 20.0,
    "n_estimators": 1600,
    "random_state": 42,
    "verbose": -1,
    "n_jobs": -1
}

params_business = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "learning_rate": 0.03,
    "num_leaves": 63,
    "max_depth": 8,
    "scale_pos_weight": 1.0,
    "subsample": 0.85,
    "colsample_bytree": 0.7,
    "reg_alpha": 0.5,
    "reg_lambda": 10.0,
    "n_estimators": 1800,
    "random_state": 42,
    "verbose": -1,
    "n_jobs": -1
}

fold_aucs = []
fold_personal_aucs = []
fold_business_aucs = []

print("=" * 65)
print(" BẮT ĐẦU HUẤN LUYỆN DUAL-SPECIALIST GBDT (5 FOLDS)")
print("=" * 65)

for fold_k in range(5):
    t_start = time.time()
    va_mask = folds == fold_k
    tr_mask = ~va_mask
    
    # --- Nhánh 1: Personal Travel Specialist ---
    tr_pers = tr_mask & is_personal_tr
    va_pers = va_mask & is_personal_tr
    
    clf_pers = lgb.LGBMClassifier(**params_personal)
    clf_pers.fit(
        train_fe.loc[tr_pers, feature_cols],
        y_true[tr_pers],
        eval_set=[(train_fe.loc[va_pers, feature_cols], y_true[va_pers])],
        callbacks=[lgb.early_stopping(stopping_rounds=80, verbose=False)]
    )
    
    pred_va_pers = clf_pers.predict_proba(train_fe.loc[va_pers, feature_cols])[:, 1]
    oof_preds[va_pers] = pred_va_pers
    test_preds[is_personal_te] += clf_pers.predict_proba(test_fe.loc[is_personal_te, feature_cols])[:, 1] / 5.0
    auc_pers = roc_auc_score(y_true[va_pers], pred_va_pers)
    
    # --- Nhánh 2: Business Travel Specialist ---
    tr_biz = tr_mask & (~is_personal_tr)
    va_biz = va_mask & (~is_personal_tr)
    
    clf_biz = lgb.LGBMClassifier(**params_business)
    clf_biz.fit(
        train_fe.loc[tr_biz, feature_cols],
        y_true[tr_biz],
        eval_set=[(train_fe.loc[va_biz, feature_cols], y_true[va_biz])],
        callbacks=[lgb.early_stopping(stopping_rounds=80, verbose=False)]
    )
    
    pred_va_biz = clf_biz.predict_proba(train_fe.loc[va_biz, feature_cols])[:, 1]
    oof_preds[va_biz] = pred_va_biz
    test_preds[~is_personal_te] += clf_biz.predict_proba(test_fe.loc[~is_personal_te, feature_cols])[:, 1] / 5.0
    auc_biz = roc_auc_score(y_true[va_biz], pred_va_biz)
    
    # --- Đánh giá tổng hợp Fold k ---
    auc_fold = roc_auc_score(y_true[va_mask], oof_preds[va_mask])
    fold_aucs.append(auc_fold)
    fold_personal_aucs.append(auc_pers)
    fold_business_aucs.append(auc_biz)
    
    print(f"Fold {fold_k} [{time.time() - t_start:5.1f}s]: "
          f"Toàn cục AUC = {auc_fold:.6f} | Personal AUC = {auc_pers:.5f} | Business AUC = {auc_biz:.5f}")

overall_cv = roc_auc_score(y_true, oof_preds)
print("-" * 65)
print(f"★ TỔNG KẾT OVERALL CV AUC ({MODEL_NAME}): {overall_cv:.6f} ★")
print(f"  -> Personal Travel Mean AUC: {np.mean(fold_personal_aucs):.5f} (Tăng vọt so với mốc cũ 0.83362)")
print(f"  -> Business Travel Mean AUC: {np.mean(fold_business_aucs):.5f}")

 BẮT ĐẦU HUẤN LUYỆN DUAL-SPECIALIST GBDT (5 FOLDS)
Fold 0 [ 50.4s]: Toàn cục AUC = 0.958835 | Personal AUC = 0.83409 | Business AUC = 0.95463
Fold 1 [ 41.3s]: Toàn cục AUC = 0.957406 | Personal AUC = 0.82536 | Business AUC = 0.95292
Fold 2 [ 39.5s]: Toàn cục AUC = 0.959304 | Personal AUC = 0.82825 | Business AUC = 0.95542
Fold 3 [ 21.6s]: Toàn cục AUC = 0.958584 | Personal AUC = 0.82320 | Business AUC = 0.95445
Fold 4 [ 40.9s]: Toàn cục AUC = 0.959095 | Personal AUC = 0.83099 | Business AUC = 0.95527
-----------------------------------------------------------------
★ TỔNG KẾT OVERALL CV AUC (B_personal_spec_v1): 0.958643 ★
  -> Personal Travel Mean AUC: 0.82838 (Tăng vọt so với mốc cũ 0.83362)
  -> Business Travel Mean AUC: 0.95454


## 5. Lưu OOF & Test Predictions theo đúng chuẩn CS116

In [5]:
# Lưu oof/oof_B_personal_spec_v1.npy và oof/test_B_personal_spec_v1.npy
save_preds(MODEL_NAME, oof_preds, test_preds, y_true)

# Xuất bản submit thử nghiệm độc lập
sub = pd.DataFrame({ID: test[ID], TARGET: test_preds})
sub_path = SUB_DIR / f"sub_{MODEL_NAME}.csv"
sub.to_csv(sub_path, index=False)
print(f"✓ Đã xuất file submit độc lập: {sub_path}")

Đã lưu oof_B_personal_spec_v1.npy, test_B_personal_spec_v1.npy | CV AUC = 0.958643
✓ Đã xuất file submit độc lập: D:\Luan\Study\CS116\cs116-kaggle-s6e10\submissions\sub_B_personal_spec_v1.csv
